# Minimal LoRA XLM-R + OOF Noise Weighting

Fast pilot for noisy/ambiguous-sample downweighting with `xlm-roberta-base`.

Plan: train only the first stratified OOF fold for `0.5` epoch, get OOF probabilities on that held-out fold, mark the top `q%` highest OOF-error examples as noisy for `q in {5, 10, 15}`, then run final weighted training. Samples without pilot OOF predictions keep weight `1.0`.

In [ ]:
# Run once on the cluster if needed.
# %pip install -q -U "transformers>=4.40" datasets accelerate "peft>=0.10" scikit-learn

In [ ]:
from pathlib import Path
import gc
import inspect
import os
import random
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, Value
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import StratifiedKFold, train_test_split
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, set_seed

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SEED = 42
MODEL_ID = "xlm-roberta-base"
TRAIN_CSV = ROOT / "data" / "train_lang.csv"
TEST_CSV = ROOT / "data" / "test.csv"
OUTPUT_DIR = ROOT / "outputs" / "minimal_lora_noise_weighting_xlmr"

# Use e.g. 20000 for a smoke test; None for full data.
SAMPLE_N = None
VAL_SIZE = 0.10
MAX_LENGTH = 128
N_CLASSES = 5

# Fast pilot: one OOF fold, half epoch. Increase FOLDS_TO_RUN for real OOF coverage.
OOF_N_SPLITS = 5
FOLDS_TO_RUN = 1
OOF_EPOCHS = 0.5

# Final weighted runs.
Q_VALUES = [5, 10, 15]
NOISY_WEIGHT = 0.5
FINAL_EPOCHS = 0.5

BATCH_SIZE = 64
EVAL_BATCH_SIZE = 1024
LR = 1.5e-4
FP16 = torch.cuda.is_available()

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05

os.environ.setdefault("WANDB_DISABLED", "true")
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## Data

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df["sentence"] = df["sentence"].fillna("")

if SAMPLE_N is not None and SAMPLE_N < len(df):
    df, _ = train_test_split(df, train_size=SAMPLE_N, random_state=SEED, stratify=df["label"])

train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=SEED, stratify=df["label"])
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train", train_df.shape, "val", val_df.shape)
print(train_df["label"].value_counts().sort_index().to_dict())
train_df.head()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def tokenize(batch):
    out = tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)
    out["labels"] = [int(x) for x in batch["label"]]
    if "sample_weight" in batch:
        out["sample_weight"] = [float(x) for x in batch["sample_weight"]]
    return out


def to_dataset(frame):
    ds = Dataset.from_pandas(frame, preserve_index=False)
    ds = ds.map(tokenize, batched=True, remove_columns=ds.column_names)
    ds = ds.cast_column("labels", Value("int64"))
    if "sample_weight" in ds.column_names:
        ds = ds.cast_column("sample_weight", Value("float32"))
    ds.set_format("torch")
    return ds


val_ds = to_dataset(val_df)

## Model And Metrics

In [ ]:
def make_model():
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=N_CLASSES)
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["query", "key", "value", "intermediate.dense", "output.dense"],
        modules_to_save=["classifier"],
        lora_dropout=LORA_DROPOUT,
        task_type=TaskType.SEQ_CLS,
    )
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")
    return model


def softmax_np(logits):
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp = np.exp(logits)
    return exp / exp.sum(axis=1, keepdims=True)


def expected_mae_risk(probs):
    classes = np.arange(probs.shape[1])
    return np.stack([np.sum(probs * np.abs(pred - classes), axis=1) for pred in classes], axis=1)


def bayes_mae_decode(probs):
    return expected_mae_risk(probs).argmin(axis=1).astype(int)


def ce_loss_np(probs, labels, eps=1e-12):
    labels = np.asarray(labels, dtype=int)
    return -np.log(np.clip(probs[np.arange(len(labels)), labels], eps, 1.0))


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax_np(logits)
    labels = np.asarray(labels, dtype=int).reshape(-1)
    map_preds = probs.argmax(axis=1).astype(int)
    bayes_preds = bayes_mae_decode(probs)
    expected_score = probs @ np.arange(N_CLASSES)
    return {
        "accuracy": float(accuracy_score(labels, map_preds)),
        "map_mae": float(mean_absolute_error(labels, map_preds)),
        "bayes_mae": float(mean_absolute_error(labels, bayes_preds)),
        "expected_score_mae": float(mean_absolute_error(labels, expected_score)),
    }


def make_training_args(**kwargs):
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in params and "evaluation_strategy" in kwargs:
        kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
    return TrainingArguments(**kwargs)

## Pilot OOF Fold

In [ ]:
oof_probs = np.full((len(train_df), N_CLASSES), np.nan, dtype=np.float32)
oof_fold = np.full(len(train_df), -1, dtype=int)

skf = StratifiedKFold(n_splits=OOF_N_SPLITS, shuffle=True, random_state=SEED)
for fold, (fit_idx, oof_idx) in enumerate(skf.split(train_df["sentence"], train_df["label"])):
    if fold >= FOLDS_TO_RUN:
        break

    print(f"OOF fold {fold}: fit={len(fit_idx)} oof={len(oof_idx)} epochs={OOF_EPOCHS}")
    fold_train_ds = to_dataset(train_df.iloc[fit_idx].reset_index(drop=True))
    fold_oof_ds = to_dataset(train_df.iloc[oof_idx].reset_index(drop=True))

    args = make_training_args(
        output_dir=str(OUTPUT_DIR / "oof_checkpoints" / f"fold_{fold}"),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        num_train_epochs=OOF_EPOCHS,
        evaluation_strategy="no",
        logging_steps=100,
        save_strategy="no",
        fp16=FP16,
        report_to=[],
        remove_unused_columns=False,
        seed=SEED + fold,
    )

    trainer = Trainer(
        model=make_model(),
        args=args,
        train_dataset=fold_train_ds,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    oof_probs[oof_idx] = softmax_np(trainer.predict(fold_oof_ds).predictions).astype(np.float32)
    oof_fold[oof_idx] = fold

    del trainer, fold_train_ds, fold_oof_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

available_oof = np.isfinite(oof_probs).all(axis=1)
print("OOF-covered samples:", int(available_oof.sum()), "/", len(train_df))

## Noise Scores And Weights

In [ ]:
train_labels = train_df["label"].to_numpy(dtype=int)
oof_map_preds = np.full(len(train_df), -1, dtype=int)
oof_bayes_preds = np.full(len(train_df), -1, dtype=int)
oof_abs_error = np.full(len(train_df), np.nan, dtype=np.float32)
oof_ce_loss = np.full(len(train_df), np.nan, dtype=np.float32)

oof_map_preds[available_oof] = oof_probs[available_oof].argmax(axis=1).astype(int)
oof_bayes_preds[available_oof] = bayes_mae_decode(oof_probs[available_oof])
oof_abs_error[available_oof] = np.abs(oof_bayes_preds[available_oof] - train_labels[available_oof])
oof_ce_loss[available_oof] = ce_loss_np(oof_probs[available_oof], train_labels[available_oof])

oof_diagnostics = train_df.copy()
oof_diagnostics["oof_fold"] = oof_fold
oof_diagnostics["oof_map_pred"] = oof_map_preds
oof_diagnostics["oof_bayes_pred"] = oof_bayes_preds
oof_diagnostics["oof_abs_error"] = oof_abs_error
oof_diagnostics["oof_ce_loss"] = oof_ce_loss
for k in range(N_CLASSES):
    oof_diagnostics[f"oof_p_{k}"] = oof_probs[:, k]

covered = oof_diagnostics[oof_diagnostics["oof_fold"] >= 0].copy()
print("pilot OOF MAP MAE:", mean_absolute_error(covered["label"], covered["oof_map_pred"]))
print("pilot OOF Bayes MAE:", mean_absolute_error(covered["label"], covered["oof_bayes_pred"]))
covered.sort_values(["oof_abs_error", "oof_ce_loss"], ascending=False).head(10)

In [ ]:
def weights_for_q(q_percent, noisy_weight=NOISY_WEIGHT):
    weights = np.ones(len(train_df), dtype=np.float32)
    noisy = np.zeros(len(train_df), dtype=bool)
    covered_idx = np.flatnonzero(available_oof)
    n_noisy = max(1, int(np.ceil(len(covered_idx) * q_percent / 100)))

    # Sort by absolute OOF Bayes error, then CE loss to break ties among equal ordinal errors.
    order = np.lexsort((-oof_ce_loss[covered_idx], -oof_abs_error[covered_idx]))
    noisy_idx = covered_idx[order[:n_noisy]]
    noisy[noisy_idx] = True
    weights[noisy_idx] = noisy_weight
    return weights, noisy


weight_summaries = []
for q in Q_VALUES:
    weights, noisy = weights_for_q(q)
    weight_summaries.append({
        "q_percent": q,
        "noisy_weight": NOISY_WEIGHT,
        "n_noisy": int(noisy.sum()),
        "effective_train_weight": float(weights.sum()),
        "mean_noisy_abs_error": float(np.nanmean(oof_abs_error[noisy])),
        "mean_noisy_ce_loss": float(np.nanmean(oof_ce_loss[noisy])),
    })

weight_summary = pd.DataFrame(weight_summaries)
display(weight_summary)

## Final Weighted Training Sweep

In [ ]:
class WeightedCETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        sample_weight = inputs.pop("sample_weight", None)
        outputs = model(**inputs)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs["logits"]
        per_sample_loss = F.cross_entropy(logits, labels.long().view(-1), reduction="none")
        if sample_weight is not None:
            sample_weight = sample_weight.to(per_sample_loss.device, dtype=per_sample_loss.dtype).view(-1)
            loss = (per_sample_loss * sample_weight).sum() / sample_weight.sum().clamp_min(1e-8)
        else:
            loss = per_sample_loss.mean()
        return (loss, outputs) if return_outputs else loss


final_summaries = []
best_trainer = None
best_q = None
best_bayes_mae = np.inf

for q in Q_VALUES:
    weights, noisy = weights_for_q(q)
    weighted_train_df = train_df.copy()
    weighted_train_df["sample_weight"] = weights
    train_ds = to_dataset(weighted_train_df)

    run_dir = OUTPUT_DIR / f"final_q{q}_w{NOISY_WEIGHT:g}"
    args = make_training_args(
        output_dir=str(run_dir / "checkpoints"),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        num_train_epochs=FINAL_EPOCHS,
        evaluation_strategy="steps",
        eval_steps=500,
        logging_steps=100,
        save_strategy="no",
        fp16=FP16,
        report_to=[],
        remove_unused_columns=False,
        seed=SEED + q,
    )

    trainer = WeightedCETrainer(
        model=make_model(),
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )
    print(f"final weighted run: q={q}, noisy={int(noisy.sum())}, weight={NOISY_WEIGHT}")
    trainer.train()
    metrics = trainer.evaluate()

    final_summaries.append({
        "q_percent": q,
        "noisy_weight": NOISY_WEIGHT,
        "n_noisy": int(noisy.sum()),
        "map_mae": metrics["eval_map_mae"],
        "bayes_mae": metrics["eval_bayes_mae"],
        "accuracy": metrics["eval_accuracy"],
        "expected_score_mae": metrics["eval_expected_score_mae"],
    })
    if metrics["eval_bayes_mae"] < best_bayes_mae:
        if best_trainer is not None:
            del best_trainer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        best_trainer = trainer
        best_q = q
        best_bayes_mae = metrics["eval_bayes_mae"]
    else:
        del trainer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

summary = pd.DataFrame(final_summaries).sort_values("bayes_mae")
display(summary)

## Save Best Run Diagnostics

In [ ]:
best_q = int(best_q)

val_out = best_trainer.predict(val_ds)
val_probs = softmax_np(val_out.predictions)
val_labels = val_out.label_ids.astype(int)
map_preds = val_probs.argmax(axis=1).astype(int)
bayes_preds = bayes_mae_decode(val_probs)

val_predictions = pd.concat(
    [
        val_df.reset_index(drop=True),
        pd.Series(map_preds, name="map_pred"),
        pd.Series(bayes_preds, name="bayes_mae_pred"),
        pd.DataFrame(val_probs, columns=[f"p_{k}" for k in range(N_CLASSES)]),
    ],
    axis=1,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_trainer.save_model(str(OUTPUT_DIR / f"best_adapter_q{best_q}"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "tokenizer"))
summary.to_csv(OUTPUT_DIR / "final_weighted_summary.csv", index=False)
weight_summary.to_csv(OUTPUT_DIR / "weight_summary.csv", index=False)
oof_diagnostics.to_csv(OUTPUT_DIR / "oof_diagnostics.csv", index=False)
val_predictions.to_csv(OUTPUT_DIR / "validation_predictions_best.csv", index=False)
print("best q:", best_q)
print(OUTPUT_DIR)
val_predictions.head()

## Optional Test Submission

In [ ]:
if TEST_CSV.exists():
    test_df = pd.read_csv(TEST_CSV)
    test_df["sentence"] = test_df["sentence"].fillna("")
    test_ds_raw = Dataset.from_pandas(test_df, preserve_index=False)

    def tokenize_test(batch):
        return tokenizer(batch["sentence"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

    test_ds = test_ds_raw.map(tokenize_test, batched=True, remove_columns=test_ds_raw.column_names)
    test_ds.set_format("torch")
    test_probs = softmax_np(best_trainer.predict(test_ds).predictions)
    test_preds = bayes_mae_decode(test_probs)

    submission_path = OUTPUT_DIR / f"submission_noise_weighted_q{best_q}_bayes_mae.csv"
    pd.DataFrame({"id": test_df["id"], "label": test_preds.astype(int)}).to_csv(submission_path, index=False)
    print("counts:", np.bincount(test_preds, minlength=N_CLASSES).tolist())
    print(submission_path)
else:
    print("No test CSV found:", TEST_CSV)